==================FINAL BASE MODEL CoT BENCHMARK RESULTS==================

Correct: 850/1319

Error: 0.36%

Accuracy: 64.44%

======================================================================


==================FINAL BASE MODEL DIRECT PROMPT BENCHMARK RESULTS==================

Correct: 264/1319

Error: 0.80%

Accuracy: 20.02%

======================================================================

==================FINAL BASE MODEL BENCHMARK RESULTS ( + SELF CONSISTENCY 5 + DIRECT)==================

Correct: 267/1319

Error: 0.80%

Accuracy: 20.24%

======================================================================

==================FINAL BASE MODEL BENCHMARK RESULTS ( + SELF CONSISTENCY 5 + COT)==================

Correct: 916/1319

Error: 0.31%

Accuracy: 69.45%

======================================================================

In [1]:
from datasets import load_dataset, Dataset
from collections import Counter

/home/my_ubuntu/projects/LRM-Distil/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("open-thoughts/OpenThoughts-114k", "metadata", split="train")

In [3]:
print(len(ds))
print(ds.column_names)
math_ds = ds.filter(lambda x: x['domain'] == "math")

113957
['problem', 'deepseek_reasoning', 'deepseek_solution', 'ground_truth_solution', 'domain', 'source', 'test_cases', 'starter_code']


In [4]:
print(f"Math DS: {len(math_ds)}")
subset = math_ds.shuffle(seed=42).select(range(11000))
print(f"Subset: {len(subset)}")

Math DS: 89120
Subset: 11000


In [5]:
def convert_open_though(ds: list[dict[str, str]]) -> list[dict[str, str]]:
    messages = []

    for sample in ds:
        messages.append(
            {
                "question": sample["problem"],
                "answer": f"<think>\n{sample['deepseek_reasoning']}\n</think>\n{sample['deepseek_solution']}"
            }
        )

    return messages

In [6]:
train_ds = subset.select(range(10000))
val_ds = subset.select(range(10000, 11000))

In [ ]:
converted_train_ds = convert_open_though(train_ds)
converted_val_ds = convert_open_though(val_ds)

print(len(converted_train_ds))
print(converted_train_ds[0])

10000
{'question': 'Given a square \\(ABCD\\) with point \\(P\\) inside such that \\(PA = 1\\), \\(PB = 2\\), and \\(PC = 3\\), calculate the angle \\(\\widehat{APB}\\).', 'answer': '<think>\nOkay, so I have this geometry problem here: There\'s a square ABCD with a point P inside such that PA = 1, PB = 2, and PC = 3. I need to find the angle APB, which is the angle at point P between points A and B. Hmm, sounds a bit tricky, but let me try to work through it step by step.\n\nFirst, let me visualize the square. Let\'s label the square ABCD with A at the bottom left, B at the bottom right, C at the top right, and D at the top left. So, if it\'s a square, all sides are equal, and all angles are 90 degrees. Point P is somewhere inside the square. Now, PA is 1, PB is 2, and PC is 3. So, the distances from P to three of the square\'s vertices are given. I need to find the angle between PA and PB at point P.\n\nI remember that in geometry problems involving distances from a point to several v

In [9]:
huggingface_train_ds = Dataset.from_list(converted_train_ds)
huggingface_val_ds = Dataset.from_list(converted_val_ds)

DATA_DIR = "./data/SFT/"
huggingface_train_ds.to_json(DATA_DIR + "open_thought_10k_train.jsonl")
huggingface_val_ds.to_json(DATA_DIR + "open_thought_1k_val.jsonl")

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  5.62ba/s]


20723557

In [ ]:
!hf